# SQL 활용 데이터 분석
Kaggle에 있는 데이터셋으로 유저 지표 등을 분석한다.
단, SQL을 위주로 활용하며, 파이썬에 sql을 연동하여 사용하고자 한다.

조금 문법이 달라지는 부분이 있긴하나 그냥 진행한다. 

data source : https://www.kaggle.com/datasets/yaybeedee/multi-category-online-store-funnel

In [3]:
import sqlite3
import pandas as pd
df = pd.read_csv('C:/Users/USER/Desktop/TIL_new/data_analyst_sql/events.csv')
# 없을때는 테이블이 생성되고, 있을때는 불러온다
con = sqlite3.connect("events.db")
#최초로 테이블을 create안하고 이미 있는 파일을 load 시킬려면 이렇게 한다.
#다만 이미 생성된 이후로는 error걸리므로 최초 실행 이후로는 주석 필수!
#df.to_sql('events', con)

In [ ]:
df.head(5)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885129 entries, 0 to 885128
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     885129 non-null  object 
 1   event_type     885129 non-null  object 
 2   product_id     885129 non-null  int64  
 3   category_id    885129 non-null  int64  
 4   category_code  648910 non-null  object 
 5   brand          672765 non-null  object 
 6   price          885129 non-null  float64
 7   user_id        885129 non-null  int64  
 8   user_session   884964 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 60.8+ MB


In [7]:
# 다음과 같이 하나의 열을 직접 볼 수 있음
cur = con.cursor()
cur.execute('SELECT * FROM events')
cur.fetchone()

(0,
 '2020-09-24 11:57:06 UTC',
 'view',
 1996170,
 2144415922528452715,
 'electronics.telephone',
 None,
 31.9,
 1515915625519388267,
 'LJuJVLEjPT')

혹은 다음과 같은 방법으로도 할 수 있다.

In [5]:
script="""
SELECT *
FROM events
WHERE event_type='view'
ORDER BY price
LIMIT 1
"""

cur.execute(script)
result = cur.fetchall()
print(result)
# column이 나오게 정리할 수는 없는가?

[(262938, '2020-11-12 04:45:56 UTC', 'view', 327938, 2144415931202273614, None, None, 0.22, 1515915625538189543, 'Zq1S1SEdsh')]


다음과 같이 문제가 생긴다. 뭔 수를 써도, 컬럼명과 같이 데이터프레임 형태로 뜨지는 않아 직접 작업해야 할 것이다.

문제는, 날짜가 더이상 날짜가 아니라 문자열 형태로 들어왔음에 주의해서 짜야한다.

## MAU

우선 월간 활성화 유저 수를 구해보자. 뭔 짓을 해도 1회의 이상의 로그만 있으면 되므로 그냥 unique한 유저 수만 세주자

In [17]:
script = """
SELECT
    substr(event_time, 1, 7) as year_month,
    count(distinct user_id) as mau
FROM events
GROUP BY year_month
ORDER BY year_month
"""
cur.execute(script)
result = cur.fetchall()
print(result)

[('2020-09', 15334), ('2020-10', 84216), ('2020-11', 92600), ('2020-12', 72137), ('2021-01', 81256), ('2021-02', 74606)]


다음과 같이 월별 활성화 유저 수를 구할 수 있다.

## DAU

In [18]:
script = """
SELECT
    substr(event_time, 1, 10) as day,
    count(distinct user_id) as dau
FROM events
GROUP BY day
ORDER BY day
"""
cur.execute(script)
result = cur.fetchall()
print(result)

[('2020-09-24', 1358), ('2020-09-25', 2451), ('2020-09-26', 2030), ('2020-09-27', 2187), ('2020-09-28', 2732), ('2020-09-29', 2834), ('2020-09-30', 2734), ('2020-10-01', 2699), ('2020-10-02', 2559), ('2020-10-03', 2162), ('2020-10-04', 2398), ('2020-10-05', 2832), ('2020-10-06', 2676), ('2020-10-07', 2561), ('2020-10-08', 2661), ('2020-10-09', 2416), ('2020-10-10', 2115), ('2020-10-11', 2430), ('2020-10-12', 2978), ('2020-10-13', 2734), ('2020-10-14', 2691), ('2020-10-15', 3199), ('2020-10-16', 3488), ('2020-10-17', 2867), ('2020-10-18', 3114), ('2020-10-19', 3690), ('2020-10-20', 3606), ('2020-10-21', 3484), ('2020-10-22', 3409), ('2020-10-23', 3234), ('2020-10-24', 2963), ('2020-10-25', 3048), ('2020-10-26', 3385), ('2020-10-27', 3518), ('2020-10-28', 3608), ('2020-10-29', 3558), ('2020-10-30', 3338), ('2020-10-31', 2856), ('2020-11-01', 3052), ('2020-11-02', 3822), ('2020-11-03', 3642), ('2020-11-04', 3434), ('2020-11-05', 3687), ('2020-11-06', 3667), ('2020-11-07', 3194), ('2020-11

## Stickness

9월 말부터 2021년도의 데이터를 구할 수 있다. 또한 이를 바탕으로 2020-10 ~ 2021-02 의 stickness를 구할 수 있을 것이다.
stickness = DaU/Mau 

문제는 join 조건을 주의해서 해야 한다는 점이다. * 100으로 % 단위로 구하자.
나누기 정수만 나오는 문제 해결을 위해 +0.00을 붙여줘야 한다...

In [27]:
script="""
SELECT
    DAU.day,
    ROUND((dau+0.00)/(mau+0.00) * 100, 2) as stickness
FROM(
    SELECT
        substr(event_time, 1, 10) as day,
        count(distinct user_id) as dau
    FROM events
    GROUP BY day
    ORDER BY day) AS DAU
    LEFT JOIN 
    (
    SELECT
        substr(event_time, 1, 7) as year_month,
        count(distinct user_id) as mau
    FROM events
    GROUP BY year_month
    ORDER BY year_month) AS MAU
    ON substr(DAU.day,1,7) = MAU.year_month
WHERE year_month != '2020-09'
ORDER BY day
"""

cur.execute(script)
result = cur.fetchall()
print(result)

[('2020-10-01', 3.2), ('2020-10-02', 3.04), ('2020-10-03', 2.57), ('2020-10-04', 2.85), ('2020-10-05', 3.36), ('2020-10-06', 3.18), ('2020-10-07', 3.04), ('2020-10-08', 3.16), ('2020-10-09', 2.87), ('2020-10-10', 2.51), ('2020-10-11', 2.89), ('2020-10-12', 3.54), ('2020-10-13', 3.25), ('2020-10-14', 3.2), ('2020-10-15', 3.8), ('2020-10-16', 4.14), ('2020-10-17', 3.4), ('2020-10-18', 3.7), ('2020-10-19', 4.38), ('2020-10-20', 4.28), ('2020-10-21', 4.14), ('2020-10-22', 4.05), ('2020-10-23', 3.84), ('2020-10-24', 3.52), ('2020-10-25', 3.62), ('2020-10-26', 4.02), ('2020-10-27', 4.18), ('2020-10-28', 4.28), ('2020-10-29', 4.22), ('2020-10-30', 3.96), ('2020-10-31', 3.39), ('2020-11-01', 3.3), ('2020-11-02', 4.13), ('2020-11-03', 3.93), ('2020-11-04', 3.71), ('2020-11-05', 3.98), ('2020-11-06', 3.96), ('2020-11-07', 3.45), ('2020-11-08', 3.7), ('2020-11-09', 4.17), ('2020-11-10', 4.2), ('2020-11-11', 4.33), ('2020-11-12', 3.86), ('2020-11-13', 3.81), ('2020-11-14', 3.62), ('2020-11-15', 3.

## Funnel

다음은 퍼널이다. 데이터 컬럼명이 나오지 않아 어쩔 수 없이 원본은 그냥 파이썬으로 보자.

In [28]:
df.head(5)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,NaN,31.90,1515915625519388267,LJuJVLEjPT
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,NaN,NaN,9.81,1515915625513238515,4TMArHtXQy
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,NaN,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ


In [29]:
script = """
SELECT
    distinct(event_type)
FROM events
"""
cur.execute(script)
result = cur.fetchall()
print(result)

[('view',), ('cart',), ('purchase',)]


정확한 event_type은 3가지 이다. 여기서 퍼널은 두가지로 나뉠 것이다.

(1) 정말 단순하게 view -> cart -> purchase

문제는 상품을 여러번 많이 보고 cart로 넘기는 행위도 있을 것이다.
중간 스킵 등도 있겠지만 단순한 funnel check로는 가능할 것이다.
따라서 여기서는 그냥 줄 수만 세면 된다.

(2) 하나의 세션 내에서의 행동으로 볼 수 있다. 각 세션에서 view -> cart -> purchse가 어떻게 넘어가나를 볼 수 있다.

### simple funnel

In [30]:
script = """
SELECT
    count(event_type) as event_type
FROM events
GROUP BY event_type
ORDER BY event_type DESC
"""
cur.execute(script)
result = cur.fetchall()
print(result)

[(793748,), (54035,), (37346,)]


상품 뷰 793748 건, 카트 54035건, 구매 37346건으로 기본 퍼널이 형성됨을 알 수 있다.

### session funnel

각 세션별로 특정 이벤트가 있었는지부터 확인해보자..

In [38]:
script = """
SELECT
    user_session,
    MAX(CASE WHEN event_type = 'view' then 1 else 0 END) AS is_view,
    MAX(CASE WHEN event_type = 'cart' then 1 else 0 END) AS is_cart,
    MAX(CASE WHEN event_type = 'purchase' then 1 else 0 END) AS is_purchase
FROM events
GROUP BY user_session
"""
cur.execute(script)
result = cur.fetchmany(45)
print(result)

[(None, 1, 1, 0), ('000AMhYaQu', 1, 0, 0), ('000c34fa-991f-442a-8e07-8c472269bec6', 1, 0, 0), ('001HttdHUk', 1, 0, 0), ('001P7lK0Pt', 1, 0, 0), ('001RxUtFJa', 1, 0, 0), ('002DmERG1w', 1, 0, 0), ('003QqC0jk0', 1, 0, 0), ('003pEktS1X', 1, 0, 0), ('003syqud5i', 1, 0, 0), ('0041IWrUEl', 1, 0, 0), ('004Rm49jyA', 1, 0, 0), ('0052kqDq2o', 1, 0, 0), ('005Qdxhw4d', 1, 0, 0), ('005wDfXQrv', 1, 0, 1), ('006j6NLVsz', 1, 0, 0), ('00735945-0395-48db-967c-59418c0ddb89', 1, 1, 0), ('007NRn5JHs', 1, 0, 0), ('007otZ8Qr0', 1, 0, 0), ('008N9fJ4hP', 1, 0, 0), ('009145ee-28c5-4afc-a104-fb3787398287', 1, 0, 0), ('009ac5g9qi', 1, 0, 0), ('009so5Mgnj', 1, 0, 0), ('009zFA5YxX', 1, 1, 0), ('00ATIKSi49', 1, 0, 0), ('00AnjtXSYC', 1, 1, 0), ('00B4xBIIZJ', 1, 1, 0), ('00BbizOhNT', 1, 0, 0), ('00BmN7cGsf', 1, 0, 0), ('00CJGqAlbB', 1, 0, 0), ('00CVIGkXfc', 1, 0, 0), ('00Cb9Z13da', 1, 0, 0), ('00CbJ5oeHx', 1, 0, 0), ('00CcHZJiqz', 1, 0, 0), ('00DZS0B1W6', 1, 0, 0), ('00DnYmvtmR', 1, 0, 0), ('00EDRogZ0C', 1, 0, 0), ('00

이렇게 하면 각 세션별로, view, cart, purchase가 이루어진 횟수를 구할 수 있다. 이제 이를 바탕으로 세면 된다.

In [43]:
script = """
SELECT
    SUM(is_view) as view_count,
    SUM(is_cart) as cart_count,
    SUM(is_purchase) as purchase_count
FROM(
    SELECT
        user_session,
        MAX(CASE WHEN event_type = 'view' then 1 else 0 END) AS is_view,
        MAX(CASE WHEN event_type = 'cart' then 1 else 0 END) AS is_cart,
        MAX(CASE WHEN event_type = 'purchase' then 1 else 0 END) AS is_purchase
    FROM events
    GROUP BY user_session
)
"""
cur.execute(script)
result = cur.fetchall()
print(result)

[(488361, 41271, 24344)]


각 세션별로의 비중은 다음과 같다.

### 퍼널 분석

잠시 SQL을 내려놓고, 결과 자체를 분석해보자

- 단순 퍼널 결과 - 상품 뷰 793748건, 카트 54035건, 구매 37346건
- 세션 퍼널 결과 - 상품 뷰 488631건, 카트 41271건, 구매 24344건
물론 순서 역전 현상 등도 소수 있겠으나 이를 무시하고 진행하자

In [44]:
print(54035/793748 * 100)
print(41271/488631 * 100)

6.8075762080660365
8.446250851869817


In [46]:
print(37346/54035 * 100)
print(24344/41271 * 100)

69.11446284815398
58.98572847762351


In [47]:
# 세션 퍼널 분석
# 뷰 -> 카트
print(41271/488631 * 100)
# 카트 -> 구매
print(24344/41271 * 100)

8.446250851869817
58.98572847762351


상품 구경 후 전체 건수 기준으로는 7% 정도가, 하나의 세션에서는 8% 정도만이 뷰->카트로 넘어간다.
카트를 담은 이후로는 세션 기준으로도 50% 이상이 구매로 이어진다. 다만 50% 미만은 카트에 담아두고도 구매하지 않았다.

- 충분한 구매로 이어질 수 있도록 쿠폰 등의 마케팅 캠페인 고려 필요
- 장바구니 -> 구매까지의 퍼널 개선으로 구매 이어지게.

## Retention

리텐션은 월간 기준 구매 리텐션을 우선 짠 뒤, 월간 롤링 리텐션을 짜려고 한다.

### monthly retention

### monthy rolling retention